# 单因子回测 Demo（AShare）

文件位置：`docs/notebooks/single_factor_backtest.ipynb`。产物默认写到 `~/ashare_outputs/single_factor/`，可用环境变量 `SINGLE_FACTOR_OUT` 覆盖。

做的是：**读面板 → 算一个因子 → 看分组/多空/纯多头 Top10% 的累计 PnL 和 RankIC → 再按完整 A 股规则（成本、涨跌停、停牌、T+1、整手）跑一遍纯多头 topN 回测，输出自包含 HTML 报告**。

## 运行前提

1. WSL2 / Linux / macOS（Windows 原生不行：仓库的因子层和数据层有 POSIX 专用代码）；
2. 仓库已 `make dev`（编译好 Rust 扩展 `panelbt` / `panelops`）；
3. 数据已解压，并设置数据根：
   ```bash
   tar -xJf ashare-data-cn-20261113.tar.xz -C ~/
   export ASHARE_DATA_DIR=$HOME/ashare-data      # 该目录下应有 cn/
   ```
4. 启动 notebook（用仓库的 `.venv`）：
   ```bash
   cd AShare
   uv pip install jupyterlab ipykernel
   source .venv/bin/activate
   jupyter lab
   ```
   VS Code 远程 WSL 也可以直接把 kernel 选成 `.venv/bin/python`。

## 口径提醒

- **因子检验**（`groupbt`）：分组等权、不扣成本；信号日 t 收盘出信号，t+1 成交；用来看因子的 PnL/IC 形状；
- **策略回测**（`panelbt`）：现金/股数、整手、涨跌停、停牌、佣金、印花税、过户费、分红送转，真实成本口径；
- 因子值**越大越做多**，低波/低估值这类因子要加负号（如 `-ts_std(returns, 20)`）；
- `long_only=True` / `rank_ic=True` 是 PR #3 的新增能力（分支 `feat/groupbt-longonly-rankic`）。本 notebook 会自动探测：没有这两个参数时，用 matplotlib 画等价的 Top10%/RankIC 曲线兜底。

In [ ]:
# ===== 1. 参数与工具 =====
import json
import os
import time
from pathlib import Path

import numpy as np

import ashare.data
import ashare.report
import panelbt

assert os.environ.get("ASHARE_DATA_DIR"), "先 export ASHARE_DATA_DIR=~/ashare-data（含 cn/ 的父目录）"


class PanelSlice:
    """面板的 [start, stop) 窗口视图：给 Evaluator / groupbt / panelbt 用。"""

    def __init__(self, panel, start=0, stop=None):
        self._panel, self._start = panel, start
        self._stop = len(panel.dates) if stop is None else stop
        self.dates = np.asarray(panel.dates)[start:self._stop]
        self.symbols = np.asarray(panel.symbols)
        self.version = f"{getattr(panel, 'version', 'panel')}:{start}-{self._stop}"

    def __getitem__(self, name):
        return np.asarray(self._panel[name])[self._start:self._stop]

    @property
    def shape(self):
        return len(self.dates), len(self.symbols)


# ---- 改这里 ----
EXPRESSION = "-ts_std(returns, 20)"   # 因子表达式（平台 DSL；支持多行赋值，最后一条是信号）
UNIVERSE = "TOP1800"                  # TOP800 / TOP1800 / ALL
N_GROUPS = 10                         # 5 / 10
START = "2016-01-01"                  # 窗口起（含）
END = "2025-01-01"                    # 窗口止（不含）；默认 = 样本内
TRADE_AT = "open"                     # open / close
TOP_N = 50                            # 策略回测：纯多头持仓只数
REBALANCE = "month"                   # day / week / month / quarter / year
CAPITAL = 1e7                         # 初始资金（元）
OUT = Path(
    os.environ.get("SINGLE_FACTOR_OUT", Path.home() / "ashare_outputs" / "single_factor")
)
OUT.mkdir(parents=True, exist_ok=True)
print("输出目录:", OUT.resolve())

In [ ]:
# ===== 2. 读面板 + 求因子 =====
from ashare.factor import Evaluator, groupbt
from ashare.platform import language

FIELDS = sorted(set(groupbt.FIELDS) | {"volume", "split", "cash_dividend", "delist", "high", "low"})

data = ashare.data.open()
panel = data.panel(FIELDS)
dates = np.asarray(panel.dates)
start = int(np.searchsorted(dates, START.replace("-", "")))
stop = int(np.searchsorted(dates, END.replace("-", "")))
print(f"面板 {len(dates)} 天 × {len(panel.symbols)} 只；{dates[0]} ~ {dates[-1]}")

variants = language.parse(EXPRESSION)
if not variants:
    raise ValueError("表达式没有解析出任何变体")
print("解析后表达式:", variants[0].source)

t0 = time.perf_counter()
alpha_history = np.asarray(
    Evaluator().evaluate(variants[0].expression, PanelSlice(panel, 0, stop)), dtype=float
)
print(f"因子计算 {time.perf_counter() - t0:.1f}s，形状 {alpha_history.shape}")

alpha = np.full(panel.shape, np.nan)  # 补到全历史形状，窗口外保持 NaN
alpha[:stop] = alpha_history

listed = np.asarray(panel["is_listed"], dtype=bool)
coverage = np.isfinite(alpha).sum(1) / np.maximum(listed.sum(1), 1)
print(f"覆盖率：最新 {coverage[-1]:.1%}，回测窗口均值 {np.nanmean(coverage[start:stop]):.1%}")

In [ ]:
# ===== 3. 因子检验：分组 / 多空 / 纯多头 top 10% =====
WINDOW = (START.replace("-", ""), END.replace("-", ""))
try:
    res = groupbt.run(
        alpha, panel, UNIVERSE, n_groups=N_GROUPS, trade_at=TRADE_AT, window=WINDOW,
        rank_ic=True,
    )
    HAS_RANK_IC = getattr(res, "rank_ic", None) is not None
except TypeError:  # main 分支还没有 rank_ic 参数
    res = groupbt.run(alpha, panel, UNIVERSE, n_groups=N_GROUPS, trade_at=TRADE_AT, window=WINDOW)
    HAS_RANK_IC = False

print(res.text())
print(f"\n窗口 {res.dates[0]} ~ {res.dates[-1]}；RankIC 可用：{HAS_RANK_IC}")

In [ ]:
# ===== 4. 累计 PnL 图（SVG；没装 matplotlib 也能看）=====
from IPython.display import HTML, display

def cumsum(series):
    return np.cumsum(np.nan_to_num(np.asarray(series, dtype=float)))

try:
    svg = res.chart(long_only=True)   # PR #3：Top10% 纯多头单独一条绿线
    print("res.chart(long_only=True)：分组线 + Top 10% 纯多头 + 多空")
except TypeError:
    svg = res.chart()                 # main：分组线 + 多空
    print("当前版本 chart() 没有 long_only，图里只有分组线 + 多空（下面有等价 matplotlib 图）")
display(HTML(svg))
(OUT / "pnl.svg").write_text(svg, encoding="utf-8")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("没装 matplotlib，跳过 PNG 图；pip install matplotlib 后可出图")

if plt is not None:
    count = len(res.dates)
    x = np.arange(count)
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, series in enumerate(res.group_pnl):
        ax.plot(x, cumsum(series), lw=1, alpha=0.5,
                label=f"G{i + 1}" if i in (0, N_GROUPS - 1) else None)
    ax.plot(x, cumsum(res.pnl["long_short"]), lw=2.2, color="tab:orange", label="L/S 多空")
    ax.plot(x, cumsum(res.pnl["top"]), lw=2.0, color="tab:green", label="Top 10% 纯多头")
    ax.axhline(0, color="k", lw=0.6)
    years = [i for i in range(1, count) if str(res.dates[i])[:4] != str(res.dates[i - 1])[:4]]
    ax.set_xticks([0, *years])
    ax.set_xticklabels([str(res.dates[0])[:4], *[str(res.dates[i])[:4] for i in years]])
    ax.set_title(f"{EXPRESSION} | 累计 PnL | {UNIVERSE}")
    ax.legend(ncol=2, fontsize=9)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUT / "pnl.png", dpi=140)
    plt.show()

In [ ]:
# ===== 5. 累计 RankIC（全池/多空口径 + top 10% 组内口径）=====
if HAS_RANK_IC:
    rank_svg = res.rank_ic_chart()
    display(HTML(rank_svg))
    (OUT / "rank_ic.svg").write_text(rank_svg, encoding="utf-8")
    for name, values in res.rank_ic.items():
        mean = float(np.nanmean(values))
        std = float(np.nanstd(values))
        print(f"{name}: mean={mean:.4f}, ICIR={mean / std * np.sqrt(244):.2f}")
else:
    import panelops

    price = np.asarray(panel[TRADE_AT], dtype=float) * np.asarray(panel["adj_factor"], dtype=float)
    price = np.asarray(panelops.ts_ffill(price), dtype=float)[:stop]
    forward = np.full_like(price, np.nan)
    forward[:-1] = price[1:] / price[:-1] - 1.0
    shifted = np.full_like(alpha_history, np.nan)
    shifted[1:] = alpha_history[:-1]
    pool = np.asarray(groupbt.universe_mask(PanelSlice(panel, 0, stop), UNIVERSE))
    shifted_pool = np.zeros_like(pool)
    shifted_pool[1:] = pool[:-1]
    ic = np.asarray(panelops.rank_ic(shifted, forward, mask=shifted_pool))[start:stop]
    mean = float(np.nanmean(ic))
    std = float(np.nanstd(ic))
    print(f"全池 RankIC: mean={mean:.4f}, ICIR={mean / std * np.sqrt(244):.2f}")
    if plt is not None:
        fig, ax = plt.subplots(figsize=(12, 3.5))
        ax.plot(np.arange(len(ic)), np.cumsum(np.nan_to_num(ic)), lw=1.6, color="tab:green")
        ax.axhline(0, color="k", lw=0.6)
        ax.set_title(f"{EXPRESSION} | 累计 RankIC | {UNIVERSE}")
        ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(OUT / "rank_ic.png", dpi=140)
        plt.show()

In [ ]:
# ===== 6. 带成本 + 完整 A 股规则：纯多头 topN 回测，输出自包含 HTML 报告 =====
from ashare.backtest import market_from_panel
from ashare.platform.backtest.engine import build_targets, signal_days

window = PanelSlice(panel, start, stop)
btm = market_from_panel(data, panel=window, exec=TRADE_AT)
pool_window = groupbt.universe_mask(window, UNIVERSE)
days = signal_days(panel, start, stop, REBALANCE)
targets = build_targets(alpha[start:stop], pool_window, days, TOP_N, stop - start)
costs = btm.costs(commission_rate=2.5e-4, commission_min=5.0, slippage=5e-4)
config = btm.config(initial_cash=CAPITAL, record_trades=True, carry_blocked=True,
                    dividend_tax_rate=0.2)

t0 = time.perf_counter()
result = panelbt.run(btm.market, targets, costs, config)
print(f"panelbt.run {time.perf_counter() - t0:.1f}s | 净值 {result.nav[0]:.4f} → {result.nav[-1]:.4f}")

ts = ashare.report.tearsheet(
    result, btm.market, targets, window.dates, btm.symbols,
    data=data, benchmark="000300.SH", benchmark_kind="price",
)
print(ts)                                  # 摘要
ts.save(OUT / "cost_report.html")
print("完整报告:", (OUT / "cost_report.html").resolve())

In [ ]:
# ===== 7. 关键指标存 JSON（方便横向比较多个因子）=====
summary = {
    "expression": EXPRESSION,
    "universe": UNIVERSE,
    "window": [START, END],
    "trade_at": TRADE_AT,
    "n_groups": N_GROUPS,
    "rank_ic": {k: float(np.nanmean(v)) for k, v in (res.rank_ic or {}).items()},
    "long_short": res.to_dict()["long_short"][-1],
    "top": res.to_dict()["top"][-1],
}
(OUT / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

## 下一步 / 常见问题

- **换因子**：只改第 1 节的 `EXPRESSION` / `UNIVERSE` / `START` / `END` 重跑即可；表达式语法与网页研究平台一致（`ts_std`、`ts_rank`、`ts_corr`、`rank`、`zscore`、`neutralize`…）。
- **想看样本外**：`START="2025-01-01", END="2026-01-01"`（验证期）；`2026-01-01` 之后是留出期。
- **分组报错 `GroupingError`**：调仓日池内有因子值的股票少于分组数 / 取值太少；换大池或改 `N_GROUPS`。
- **优先看什么**：样本内 sharpe、纯多头 top 组收益、`margin`（每单位换手赚多少）、RankIC 的 ICIR；再看样本外是否衰减。
- **两个口径的差异**：第 3 节不扣成本，第 6 节按真实规则扣费；同一因子的 Sharpe 会不同，这是正常的。
- **内存**：全市场面板约 6 GB；笔记本机器建议 16 GB 以上，或先 `UNIVERSE="TOP800"`。
- `long_only` / `rank_ic` 未合并时，第 4/5 节会自动走 matplotlib 兜底；想直接用新图，checkout `feat/groupbt-longonly-rankic`。

文档：`docs/PLATFORM.md`（研究页/策略回测页）、`docs/GROUP_BACKTEST.md`（分组口径）、`docs/REPORT_NOTES.md`（报告字段）。